# modeling_v13 — M5 듀얼 재튜닝 (LGBM + XGBoost, 목적 = 정직축 GKF)

**목적**: 확정 셋 **Conservative-GA(99)** 을 고정하고, **LGBM·XGBoost** 를 각각
`stable_group_kfold(C20)` **5-fold OOF RMSE 를 목적함수**로 Optuna 튜닝 → 다중시드 lot-CV(seed 1/2/3)
스크리닝으로 트랙별 대표 1개 선발 → `tuned_params_v13_{lgbm,xgb}.json` 동결.
**모델 챔피언 선언·G5 게이트 판정은 이 노트북이 하지 않는다**(회신 숫자로 강건이 사전등록 §6.2/G5 규칙 적용).

> **왜 ET 가 없나**: M4 듀얼(REPORT_06)·XGB 다중시드(REPORT_07)에서 ET 는 정직축 최약체(lot-mate 암기),
> XGBoost 가 4파티션 전부 LGBM 우세(평균 Δ+0.590·최악 +0.426) → **PLAN v2.4 §6.2 스코프 개정(사용자 승인 07-15)**.
> ET 는 F5′ 앙상블 예비로만 동결 보존.

---
### 전제 파일 (노트북과 같은 폴더 또는 `data/`·`colab_GA/` 에 존재)
- `v13_fdc_pool_wf_oof.csv.gz` (정본 풀 11,939×659)
- `core10_meta_wf.csv` (core10 + 시간/레짐 메타)
- `feature_diet_selected.json` (champion 5센서)
- `select_result_Conservative_GA.json` (확정 셋 selected 84)

### 실행 순서 · 예상 런타임
1. **Restart & Run All**. 셀 순서대로: 상수 → 헬퍼 → 로드/floor → 목적함수 → **LGBM study** → **XGB study** → **Stage B 다중시드** → 저장.
2. 런타임(**CPU 기준**): 풀 모드 LGBM 80 trials ~40–90분 + XGB 60 trials ~30–70분 + Stage B ~20–30분 → **Colab 순조 ~1.5–3h / 로컬 ~3–6h**.
   - **빠른 스모크**: 아래 상수 셀에서 `QUICK=True` (또는 환경변수 `M5_QUICK=1`) → 6 trials·짧은 rounds 로 파이프라인만 <5분 점검.
   - XGB 는 Colab GPU 사용 시 상수 셀의 `XGB_DEVICE="cuda"` 로 대폭 단축 가능(수치는 로컬 CPU 로 재확인 = 공식, R6).

### 확인 포인트 (실행 후 강건에게 회신할 것)
- **재현 self-check**: 튜닝 전 baseline LGBM stable GKF ≈ **71.366**(Δ<0.05) 출력되는지 → 로더·폴드맵 무결 확인.
- Stage B 표의 **LGBM 대표 / XGB 대표**: `stable_GKF`, `seed_mean`, `seed_worst`, `R2`, `rounds`.
- 저장 3종: `tuned_params_v13_lgbm.json`, `tuned_params_v13_xgb.json`, `m5_stageB_summary.json` + study CSV 2개.
- **게이트는 회신 후 강건이 판정**(G5: 로컬 GKF ≤ 70.712, R²≥0.9). 이 노트북 검산 블록은 참고용.

> **정직 유의(사전 기록)**: Stage A 튜닝은 각 fold 의 조기중단(early stopping)을 **그 fold 를 eval 로** 쓴다 → OOF 에 **경미한 낙관 편향**. 그래서 **Stage B 는 고정 rounds(평균 best_iter)로 재적합**(조기중단 없음)해 정직 수치를 만들고, **챔피언·게이트는 Stage B 다중시드 수치로만** 판단한다. Colab 실행 시 Stage B 절대값은 로컬 재확인 전까지 상대비교 전용(R6).


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 동결 상수
ID_COL, TARGET_COL = "C64", "C65"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33",
          "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
SEEDS = [1, 2, 3]
SIGMA_C65 = 261.7

# baseline LGBM (M8_PARAMS 705) - 재현 self-check + 준우승 보존 기준
M8_PARAMS = dict(objective="regression", metric="rmse",
    learning_rate=0.029017547696366934, num_leaves=175, min_child_samples=5,
    feature_fraction=0.6324704159196377, bagging_fraction=0.864012693783303, bagging_freq=7,
    lambda_l1=5.04154328625296, lambda_l2=0.024814259264649002,
    min_split_gain=0.2573073648505903, verbose=-1, seed=42)
BEST_ROUNDS = 705

# 동결 참조값 (전부 로컬 venv 산; R6)
B1_LGBM_REF = 71.366
B0_REF      = 71.212
ANCHOR      = 70.712
EXPECT_STABLE_LGBM = 71.366

# 튜닝 설정
QUICK = bool(int(os.environ.get("M5_QUICK", "0")))
XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu")
if QUICK:
    N_TRIALS_LGBM, N_TRIALS_XGB = 6, 6
    MAX_ROUNDS, EARLY, TOPK = 300, 40, 2
else:
    N_TRIALS_LGBM, N_TRIALS_XGB = 80, 60
    MAX_ROUNDS, EARLY, TOPK = 3000, 100, 3
OPTUNA_SEED = 42
print(f"[설정] QUICK={QUICK} | LGBM {N_TRIALS_LGBM} / XGB {N_TRIALS_XGB} trials | "
      f"MAX_ROUNDS={MAX_ROUNDS} EARLY={EARLY} TOPK={TOPK} | XGB_DEVICE={XGB_DEVICE}")

def _find(name):
    for d in [".", "data", "colab_GA", "..",
              os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"),
              os.path.join("..", "modeling_v13", "colab_GA")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음 - 노트북과 같은 폴더(또는 data/·colab_GA/)에 두세요.")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz")
META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json")
SEL_CONS = _find("select_result_Conservative_GA.json")
print("파일 확인:", *[os.path.basename(x) for x in [POOL, META, DIET, SEL_CONS]])
print("xgboost", xgb.__version__, "| lightgbm", lgb.__version__, "| optuna", optuna.__version__)


In [ ]:
# 헬퍼
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

# v2.2 환경독립 GroupKFold (동결 stable 폴드맵)
def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]
    fold_sizes = np.zeros(n_splits, dtype=np.int64); g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fold_sizes)); fold_sizes[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]

def stable_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

# 다중시드 lot-CV: 유니크 lot 을 KFold(shuffle,seed) 로 fold 배정 -> 샘플이 자기 lot fold 상속
def lot_seed_splits(groups, seed, n_splits=5):
    groups = np.asarray(groups)
    lots = np.unique(groups)
    lot_fold = np.empty(len(lots), dtype=np.int64)
    for f, (_, te) in enumerate(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(lots)):
        lot_fold[te] = f
    l2f = {l: int(fo) for l, fo in zip(lots, lot_fold)}
    fv = np.array([l2f[g] for g in groups])
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]


In [ ]:
# 로드 & 확정 셋 Conservative-GA(99)
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
y = df[TARGET_COL].to_numpy(float)
groups = df["C20"].to_numpy()
diet = json.loads(open(DIET, encoding="utf-8").read())
champions = list(diet["champions"]["Conservative"].values())
fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
selp = json.loads(open(SEL_CONS, encoding="utf-8").read())["selected_features"]
feats = list(dict.fromkeys(fixed + selp))

ok, have = floor_ok(feats)                                  # R10
assert all(f in df.columns for f in feats), "누락 피처 존재"
assert "C20" not in feats and "C64" not in feats and "fold_kf5" not in feats, "누수 피처 유입!"
assert ok and len(feats) == 99, f"셋 문제 n={len(feats)} floor={have}"

STABLE_SPLITS = stable_splits(groups)
SEED_SPLITS = {s: lot_seed_splits(groups, s) for s in SEEDS}
print(f"Conservative-GA 확정 셋 n={len(feats)}  floor={have}")
print(f"df {df.shape} | unique C20 {len(np.unique(groups))} lot | 시드 {SEEDS}")

def _oof_lgb_fixed(params, rounds, splits):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = lgb.train(params, lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=rounds)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)
_base = _oof_lgb_fixed(M8_PARAMS, BEST_ROUNDS, STABLE_SPLITS)
_flag = "OK" if abs(_base - EXPECT_STABLE_LGBM) < 0.05 else "주의 - 환경/핀 점검"
print(f"[재현 self-check] baseline LGBM stable GKF = {_base:.3f}  "
      f"(동결 {EXPECT_STABLE_LGBM}, delta {_base-EXPECT_STABLE_LGBM:+.3f})  {_flag}")


In [ ]:
# 목적함수: stable GKF 5-fold OOF RMSE (fold별 조기중단 -> best_iter 수집)
def _prune_report(trial, y_true, oof, filled_idx, step):
    trial.report(_rmse(y_true[filled_idx], oof[filled_idx]), step)
    if trial.should_prune():
        raise optuna.TrialPruned()

def objective_lgb(trial):
    params = dict(objective="regression", metric="rmse", verbose=-1, seed=42,
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        feature_fraction=trial.suggest_float("feature_fraction", 0.4, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
        bagging_freq=trial.suggest_int("bagging_freq", 1, 7),
        lambda_l1=trial.suggest_float("lambda_l1", 1e-3, 10, log=True),
        lambda_l2=trial.suggest_float("lambda_l2", 1e-3, 10, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 0.0, 1.0))
    oof = np.zeros(len(df)); iters = []; filled = np.array([], dtype=int)
    for i, (tr, va) in enumerate(STABLE_SPLITS):
        m = lgb.train(params, lgb.Dataset(df.iloc[tr][feats], y[tr]),
                      num_boost_round=MAX_ROUNDS,
                      valid_sets=[lgb.Dataset(df.iloc[va][feats], y[va])],
                      callbacks=[lgb.early_stopping(EARLY, verbose=False), lgb.log_evaluation(0)])
        bi = m.best_iteration or MAX_ROUNDS
        oof[va] = m.predict(df.iloc[va][feats], num_iteration=bi); iters.append(bi)
        filled = np.concatenate([filled, va]).astype(int)
        _prune_report(trial, y, oof, filled, i)
    trial.set_user_attr("mean_best_iter", int(np.mean(iters)))
    return _rmse(y, oof)

def _xgb_make(params, rounds, early=None):
    kw = dict(n_estimators=rounds, tree_method="hist", device=XGB_DEVICE,
              random_state=42, n_jobs=-1, verbosity=0, **params)
    if early is not None:
        kw["early_stopping_rounds"] = early
    return xgb.XGBRegressor(**kw)

def objective_xgb(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        gamma=trial.suggest_float("gamma", 0.0, 5.0))
    oof = np.zeros(len(df)); iters = []; filled = np.array([], dtype=int)
    for i, (tr, va) in enumerate(STABLE_SPLITS):
        m = _xgb_make(params, MAX_ROUNDS, early=EARLY)
        m.fit(df.iloc[tr][feats], y[tr], eval_set=[(df.iloc[va][feats], y[va])], verbose=False)
        bi = getattr(m, "best_iteration", None)
        bi = (bi + 1) if bi is not None else MAX_ROUNDS
        try:
            oof[va] = m.predict(df.iloc[va][feats], iteration_range=(0, bi))
        except TypeError:
            oof[va] = m.predict(df.iloc[va][feats])
        iters.append(bi)
        filled = np.concatenate([filled, va]).astype(int)
        _prune_report(trial, y, oof, filled, i)
    trial.set_user_attr("mean_best_iter", int(np.mean(iters)))
    return _rmse(y, oof)
print("목적함수 준비 완료 (LGBM·XGBoost, stable GKF OOF).")


In [ ]:
# Stage A-1: LGBM Optuna (목적 = stable GKF OOF RMSE)
t0 = time.time()
study_lgb = optuna.create_study(direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS_LGBM, show_progress_bar=False)
lgb_best = study_lgb.best_trial
print(f"[LGBM] best GKF(OOF, 튜닝값) = {study_lgb.best_value:.3f} | "
      f"mean_best_iter={lgb_best.user_attrs.get('mean_best_iter')} | "
      f"완료 {len(study_lgb.trials)} trials, {time.time()-t0:.0f}s")
print("      best params:", json.dumps(lgb_best.params))
study_lgb.trials_dataframe().to_csv("m5_study_lgbm.csv", index=False)


In [ ]:
# Stage A-2: XGBoost Optuna (목적 = stable GKF OOF RMSE)
t0 = time.time()
study_xgb = optuna.create_study(direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_XGB, show_progress_bar=False)
xgb_best = study_xgb.best_trial
print(f"[XGB] best GKF(OOF, 튜닝값) = {study_xgb.best_value:.3f} | "
      f"mean_best_iter={xgb_best.user_attrs.get('mean_best_iter')} | "
      f"완료 {len(study_xgb.trials)} trials, {time.time()-t0:.0f}s")
print("      best params:", json.dumps(xgb_best.params))
study_xgb.trials_dataframe().to_csv("m5_study_xgb.csv", index=False)


In [ ]:
# Stage B: 트랙별 상위 TOPK trial 을 고정 rounds 로 4파티션(stable+seed1/2/3) 재적합
# 정직 수치(조기중단 없음). 대표 = 시드평균 최소 -> tie 시 최악 시드 우수.
def oof_lgb_fixed(params, rounds, splits):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = lgb.train({**params, "objective": "regression", "metric": "rmse",
                       "verbose": -1, "seed": 42},
                      lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=rounds)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def oof_xgb_fixed(params, rounds, splits):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = _xgb_make(params, rounds, early=None)
        m.fit(df.iloc[tr][feats], y[tr])
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def top_trials(study, k):
    done = [t for t in study.trials if t.value is not None]
    return sorted(done, key=lambda t: t.value)[:k]

def screen(track, study, oof_fixed):
    rows = []
    for rank, t in enumerate(top_trials(study, TOPK), 1):
        rounds = int(t.user_attrs.get("mean_best_iter") or BEST_ROUNDS)
        st = oof_fixed(t.params, rounds, STABLE_SPLITS)
        sd = [oof_fixed(t.params, rounds, SEED_SPLITS[s]) for s in SEEDS]
        rows.append(dict(track=track, rank=rank, trial=t.number, rounds=rounds,
                         tune_val=round(t.value, 3), stable_GKF=round(st, 3),
                         seed_mean=round(float(np.mean(sd)), 3),
                         seed_worst=round(float(np.max(sd)), 3),
                         R2=r2_honest(st), params=t.params))
        print(f"  {track} #{rank} trial{t.number}: stable={st:.3f} "
              f"seed_mean={np.mean(sd):.3f} worst={np.max(sd):.3f} rounds={rounds}")
    rows.sort(key=lambda r: (r["seed_mean"], r["seed_worst"]))
    return rows

print("=== Stage B 다중시드 스크리닝 (고정 rounds) ===")
lgb_rows = screen("LGBM", study_lgb, oof_lgb_fixed)
xgb_rows = screen("XGBoost", study_xgb, oof_xgb_fixed)
lgb_rep, xgb_rep = lgb_rows[0], xgb_rows[0]
stageB = pd.DataFrame(lgb_rows + xgb_rows,
    columns=["track","rank","trial","rounds","tune_val","stable_GKF","seed_mean","seed_worst","R2"])
stageB


In [ ]:
# 검산 (§6.2/G5 - 참고. 공식은 강건이 회신 숫자로) + 저장
def cmp_model(a, b):   # |delta|>=0.3 낮은쪽, 아니면 tie
    return ("LGBM" if a < b else "XGBoost") if abs(a - b) >= 0.3 else "tie"

L, X = lgb_rep, xgb_rep
by_gkf = cmp_model(L["stable_GKF"], X["stable_GKF"])
if by_gkf == "tie":
    l_spread = abs(L["seed_worst"] - L["seed_mean"]); x_spread = abs(X["seed_worst"] - X["seed_mean"])
    champ = "LGBM" if l_spread <= x_spread else "XGBoost"
    why = f"stable delta<0.3 tie -> 다중시드 분산 작은 쪽 = {champ} (동률이면 LGBM)"
else:
    champ = by_gkf; why = f"stable GKF |delta|>=0.3 낮은 쪽 = {champ}"
rep = L if champ == "LGBM" else X

gap = rep["stable_GKF"] - ANCHOR
anchor_str = "<= 앵커 통과권" if gap <= 0 else ("앵커 위 (+%.3f)" % gap)
guard_str = "통과" if min(L["R2"], X["R2"]) >= 0.9 else "주의 - 기각 구성 있음"
print("=== 검산 (참고) ===")
print(f"  LGBM 대표 : stable={L['stable_GKF']:.3f} seed_mean={L['seed_mean']:.3f} worst={L['seed_worst']:.3f} R2={L['R2']}")
print(f"  XGB  대표 : stable={X['stable_GKF']:.3f} seed_mean={X['seed_mean']:.3f} worst={X['seed_worst']:.3f} R2={X['R2']}")
print(f"  -> 검산 챔피언 후보 = {champ}  ({why})")
print(f"  G5 앵커 {ANCHOR}: {champ} stable={rep['stable_GKF']:.3f} -> {anchor_str}")
print(f"  가드레일 R2>=0.9: LGBM {L['R2']} · XGB {X['R2']} -> {guard_str}")
print(f"  기준선 B1_LGBM(동결)={B1_LGBM_REF} · B0={B0_REF} · 앵커={ANCHOR}")
print("  ※ 공식 셋/챔피언/게이트는 회신 숫자로 강건이 사전등록 규칙 적용.")

for name, r in [("lgbm", L), ("xgb", X)]:
    json.dump(dict(track=r["track"], params=r["params"], n_estimators=r["rounds"],
                   set="Conservative-GA(99)", n_feats=len(feats),
                   stable_GKF=r["stable_GKF"], seed_mean=r["seed_mean"],
                   seed_worst=r["seed_worst"], R2_honest=r["R2"],
                   objective="stable_group_kfold(C20) 5-fold OOF RMSE",
                   cv="v2.2 stable 폴드맵 + 다중시드 lot-CV(seed 1/2/3)",
                   note="Stage B 고정-rounds 정직 수치. Colab 산이면 로컬 재확인 = 공식(R6)."),
              open(f"tuned_params_v13_{name}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)

json.dump(dict(set="Conservative-GA(99)", n_feats=len(feats),
               baseline_selfcheck=round(_base, 3), quick=QUICK,
               lgbm_rep={k: L[k] for k in ["trial","rounds","stable_GKF","seed_mean","seed_worst","R2"]},
               xgb_rep={k: X[k] for k in ["trial","rounds","stable_GKF","seed_mean","seed_worst","R2"]},
               check_champion=champ, check_rule=why,
               gate=dict(anchor=ANCHOR, B1_LGBM=B1_LGBM_REF, B0=B0_REF,
                         champ_stable=rep["stable_GKF"], pass_anchor=bool(rep["stable_GKF"] <= ANCHOR)),
               stageB=lgb_rows + xgb_rows,
               note="공식 챔피언/G5 판정은 강건이 회신 숫자로. 이 파일은 검산·산출 인덱스."),
          open("m5_stageB_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
stageB.to_csv("m5_stageB_results.csv", index=False)
print("saved: tuned_params_v13_lgbm.json / tuned_params_v13_xgb.json / m5_stageB_summary.json "
      "/ m5_stageB_results.csv / m5_study_lgbm.csv / m5_study_xgb.csv")


---
### 실행 후 (강건 판정 순서)
1. **정합 확인**: `baseline_selfcheck` ≈ 71.366(Δ<0.05)? study CSV·요약 JSON 수치 대조.
2. **§6.2 모델 챔피언**: 로컬 GKF 낮은 쪽(|Δ|≥0.3) → 다중시드 분산 작은 쪽 → tie면 LGBM. 준우승·ET 동결본은 F5′ 예비.
3. **G5 게이트**: 챔피언 로컬 GKF **≤ 70.712**(R²≥0.9). 70.712~71.012 = 조건부. 초과 = 재시도 1회(LGBM depth/dart·XGB gamma/min_child 확장) → 미달 시 M7.
4. **문서**: README 로그 1행 + `REPORT/modeling_v13_REPORT_08_M5.md`(챔피언 선언). 통과 시 M6 배터리(챔피언 모델만), 미달 시 M7 사다리(F1′→F5′).

> **Colab 로 Stage A/B 를 돌렸다면**: `tuned_params_*.json` 의 **params 는 채택**하되, `stable_GKF`·`seed_*` **절대값은 로컬 재실행으로 확정**(R6). XGB 는 로컬 xgboost 버전 핀 필수(§10.6 P0-3, 드리프트 +0.01~0.26 확인).
